# 📝 지식 그래프 과제 LV2(응용): 배치 집계·정제·온톨로지 주입 추출

> LV1 에서 익힌 것들을 **조합**합니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 배치 진단**: 여러 문서에 걸친 관계별 집계 · 문서별 허용 관계 비율로 "다시 뽑을 문서" 고르기
> - **2. 정제와 적재 준비**: 필터와 중복 제거를 잇기 · 시그니처로 방향까지 검사 · id 해소와 `:Candidate` 격리 · `MERGE` 문 만들기 · 타입 계층 얹기
> - **3. 모델로 뽑기**: 온톨로지를 주입한 프롬프트로 실제 추출하고 그 자리에서 정제 · CoT 프롬프트 설계
> - **4. 설계 판단**: 방향 고정과 후보 격리를 자기 말로 (서술형)

## 풀이 방법
1. 맨 위 **준비 셀 네 개**를 먼저 실행하세요.
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: `data/pgx_lv2_triples.jsonl`(자가면역 질환 논문 **세 편**에 걸친 배치 추출 결과), `data/pgx_lv2_corpus.txt`(라이브 추출용 발췌 한 편, **PMC13493376**).
- 라이브 추출은 결과 문구가 매번 조금씩 달라, 채점은 **구조·개체 포함·건수 범위**만 봅니다.

화이팅!

> **데이터 출처**
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 본문 발췌 (`pgx_*.jsonl`) | PubMed Central Open Access Subset (pmcid 를 각 문서에 적어 두었습니다) | CC BY |
> | 이름 -> id 사전 (`name2id.json`) | Hetionet v1.0 (https://het.io) + RxNav(NLM) 약물 동의어 | CC0 / 공개 |
> | 큐레이션 관계 (`hetionet_curated.jsonl`) | Hetionet v1.0 에서 CC0 출처만 골라낸 부분 | CC0 |
>
> 논문에서 뽑은 관계는 **그 논문이 그렇게 보고했다**는 뜻이고, Hetionet 관계는 **2016년에 정리된** 문헌 근거라는 뜻입니다. 둘 다 "효능이 입증됐다"는 말이 아닙니다. 이 구분을 트리플 속성 `evidence_level` 로 남기는 법은 교안에서 다룹니다.

아래 준비 셀 네 개를 차례로 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 온톨로지: 실행만 하세요
# 관계 규격을 모아 두는 한 곳. 관계를 더하거나 고칠 때는 여기만 손댄다
# 각 관계: (주어 타입, 목적어 타입, 판정 기준)
RELATION_SIGNATURES = {
    "TREATS":           ("Compound", "Disease",
                         "약이 질병을 치료한다. "
                         "질병의 원인이나 진행 자체에 작용한다"),
    "PALLIATES":        ("Compound", "Disease",
                         "약이 질병의 증상을 완화한다. "
                         "질병 자체는 그대로 두고 증상만 덜어 준다"),
    "BINDS":            ("Compound", "Gene",
                         "약이 그 유전자의 단백질에 결합한다. "
                         "그 유전자가 이 약의 대사·수송을 맡는다는 진술도 여기에 적는다. "
                         "그 대신 표적·효소·수송체는 가르지 않는다"),
    "UPREGULATES_CG":   ("Compound", "Gene", "약이 그 유전자의 발현을 증가시킨다"),
    "DOWNREGULATES_CG": ("Compound", "Gene", "약이 그 유전자의 발현을 감소시킨다"),
    "ASSOCIATES":       ("Disease", "Gene", "질병과 유전자 사이에 연관이 보고됐다"),
    "PRESENTS":         ("Disease", "Symptom", "질병이 그 증상으로 나타난다"),
    "INCLUDES":         ("PharmacologicClass", "Compound",
                         "약효 분류가 그 약물을 포함한다. "
                         "그 약이 어느 계열에 속한다는 진술을 여기에 적는다"),
}

# 노드 타입 5종. 지식그래프를 적재한 단원의 레이블과 글자까지 같아야 한다
NODE_TYPES = {"Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"}

In [ ]:
# [제공 코드] 이름 -> id 사전 준비: 이 셀은 실행만 하세요.
# 사전을 어떻게 만드는지는 엔티티 정규화 단원에서 배웁니다. 여기서는 읽어 쓰기만 합니다.
import json
import re
from pathlib import Path

NAME2ID = json.loads(Path("data/name2id.json").read_text(encoding="utf-8"))


# 표기 차이를 눌러 사전 조회용 키로 만든다. lookup_id 함수가 호출한다
def normalize_name(name):
    """이름 매칭용 정규화: 소문자, 앞뒤 공백 제거, 연속 공백 한 칸, 끝의 괄호 주석 제거."""
    s = re.sub(r"\s+", " ", name).strip().lower()
    return re.sub(r"\s*\([^)]*\)$", "", s).strip()


# 이름을 노드 id 로 바꿔 돌려준다. 트리플을 그래프에 적재할 때 쓴다
def lookup_id(name, node_type):
    """이름과 타입으로 Hetionet id 를 찾아 (id, 사유) 두 칸으로 돌려준다.

    사유는 셋 중 하나다. 못 붙은 이유가 둘로 갈리므로 id 만으로는 구별할 수 없다.
      "hit"        붙었다. 첫 칸이 그 id 다
      "miss"       사전에 그 이름이 없다(또는 타입이 다르다). 첫 칸은 None
      "ambiguous"  후보가 둘 이상이라 이름만으로는 못 고른다. 첫 칸은 None
    """
    if node_type == "Gene":
        # 유전자 기호는 대소문자가 곧 뜻이다. 소문자로 누르면 CAT·SET 같은 흔한 단어가 유전자로 잡힌다
        found = NAME2ID["genes"].get(name.strip())
        return (found, "hit") if found else (None, "miss")
    key = normalize_name(name)
    if key in NAME2ID["ambiguous"]:
        return None, "ambiguous"      # 한 이름이 서로 다른 타입 두 곳에 걸린 경우
    entry = NAME2ID["entries"].get(key)
    # 타입까지 맞아야 같은 개체다. obesity 는 Disease 이면서 Symptom 이라 타입을 안 보면 엉뚱하게 붙는다
    if entry and entry["label"] == node_type:
        return entry["id"], "hit"
    return None, "miss"


print("사전 항목:", len(NAME2ID["entries"]),
      "/ 유전자 기호:", len(NAME2ID["genes"]),
      "/ 애매한 이름:", len(NAME2ID["ambiguous"]))


In [ ]:
# [제공 코드] 라이브 추출 준비: 실행만 하세요(여기서부터 모델을 실제로 부릅니다).
from typing import Literal

from pydantic import BaseModel, Field

# 온톨로지 dict 에서 값 집합을 그대로 가져온다. 관계를 더하면 서식도 따라 넓어진다
RelationName = Literal[tuple(RELATION_SIGNATURES)]
NodeType = Literal[tuple(sorted(NODE_TYPES))]   # 집합은 순서가 없어 sorted 로 고정한다


# 트리플 서식: 필드 이름이 곧 모델이 채울 칸이 된다(칸 이름을 바꾸면 모델 답의 칸도 바뀐다)
class Triple(BaseModel):
    # Field 의 description 은 사람용 주석이 아니라 모델에게 그대로 전달되는 지시다
    subject: str           = Field(description="주어. 논문에 적힌 표기 그대로")
    subject_type: NodeType = Field(description="주어 타입")
    relation: RelationName = Field(description="관계. 온톨로지의 허용 관계 중 하나")
    object: str            = Field(description="목적어. 논문에 적힌 표기 그대로")
    object_type: NodeType  = Field(description="목적어 타입")
    # 근거를 200자로 묶는다. 문단을 통째로 담으면 "이 문장이 이 트리플을 입증하는가"를 못 따진다
    evidence: str          = Field(description="근거가 된 원문 (200자 이내)")


# 문서 하나에서 사실이 여러 개 나오므로 트리플을 리스트로 받는다
# 모델에 넘길 서식은 이 바깥 클래스다(안의 Triple 은 리스트 원소의 틀)
class Extraction(BaseModel):
    triples: list[Triple] = Field(description="문서에서 뽑은 트리플 목록")


def build_ontology_block(sigs):
    """시그니처 dict 를 프롬프트에 끼울 텍스트 블록으로 바꾼다."""
    lines = ["[허용 관계]"]
    for rel, (subj, obj, crit) in sigs.items():
        # 판정 기준을 # 뒤에 함께 적는다. 모델이 관계를 고를 때 읽는 설명이 된다
        lines.append(f"- {rel}: ({subj}) -> ({obj})  # {crit}")
    return "\n".join(lines)


from langchain_core.prompts import ChatPromptTemplate


def build_extraction_prompt():
    """역할·온톨로지·규칙을 담은 추출 프롬프트 템플릿을 돌려준다(문서는 실행할 때 넣는다)."""
    # dict 를 고치면 이 블록이 따라 바뀌고, 템플릿도 저절로 최신 규격이 된다
    ontology = build_ontology_block(RELATION_SIGNATURES)
    system = ("너는 의학 논문에서 트리플을 뽑는 도구야.\n\n"
              f"{ontology}\n\n"
              "[규칙]\n"
              "- 관계는 허용 관계 중 하나만 쓰고, 방향(주어 타입 -> 목적어 타입)을 지킨다.\n"
              # 서식이 관계 이름을 허용 목록으로 좁혀 두었으니, 맞는 것이 없을 때 어떻게 할지를 규칙이 정한다.
              # 이 줄이 없으면 맞는 관계가 없을 때 모델이 목록 안에서 아무거나 고른다
              "- 맞는 관계가 없으면 그 사실은 아예 넣지 않는다. 억지로 고르지 않는다.\n"
              "- 서술문을 통째로 넣지 말고 개체 이름만 넣는다. 이름은 논문 표기 그대로 쓴다.\n"
              "- 근거(evidence)는 원문을 200자 이내로 그대로 인용한다.")
    # {text} 만 변수로 남긴다. 실행할 때 invoke({"text": ...}) 로 채운다
    return ChatPromptTemplate.from_messages([("system", system), ("human", "[문서]\n{text}")])


LIVE_TEXT = Path("data/pgx_lv2_corpus.txt").read_text(encoding="utf-8")
extraction_chain = build_extraction_prompt() | make_model().with_structured_output(Extraction)
print('라이브 추출용 발췌:', len(LIVE_TEXT), '자')
print(LIVE_TEXT[:120], '...')

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 배치 추출 결과를 훑어봅니다.

In [ ]:
# [제공 코드] 배치 추출 결과를 먼저 살펴봅니다
preview = [json.loads(line) for line in Path('data/pgx_lv2_triples.jsonl').read_text(encoding='utf-8').splitlines()]
print('행 수:', len(preview), '/ 문서:', sorted({r['doc_id'] for r in preview}))
for r in preview[:3]:
    print(r)

---
# 1. 배치 진단

여러 문서에서 한꺼번에 뽑은 결과를 관계 축과 문서 축으로 집계해, 어느 문서를 다시 뽑아야 할지 판단합니다(문서별로 결과를 세는 자리는 교안_02 3-1 과 🚀 종합 클론코딩).

## 1-1. 배치 로드하고 관계별로 집계하기
**배경**: 여러 문서에서 뽑은 트리플을 한꺼번에 불러와, 관계별로 몇 개인지 집계합니다(로드 + 집계).

**요구사항**:
- `data/pgx_lv2_triples.jsonl` 을 dict 리스트 **`batch`** 로 불러오세요.
- `relation` 별 개수를 사전 **`by_rel`** 에 담으세요(키=관계, 값=개수).

**예시**: `by_rel['TREATS']` 는 **8**, `by_rel['PRESENTS']` 는 **3** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- LV1 의 로드 방법 + 값별 빈도 집계를 이어 붙인다.

세부구현:
1. 각 줄을 json.loads 해 batch 리스트로 만든다.
2. 각 행의 relation 값만 흘려 넣어 빈도를 세고, 그 결과를 dict 로 by_rel 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 채점이 학생 답안의 import 에 기대지 않게 여기서 직접 불러온다(사전으로 세도 통과해야 한다)
from collections import Counter

assert len(batch) == 13, 'jsonl 의 모든 줄을 읽었는지 확인하세요(13줄입니다)'
assert by_rel['TREATS'] == 8 and by_rel['PRESENTS'] == 3, \
    '관계 이름을 키로, 개수를 값으로 담았는지 확인하세요'
assert by_rel['BINDS'] == 1, '정제 전 원출력을 그대로 세야 합니다'
assert all(isinstance(r, dict) and 'relation' in r for r in batch), \
    'batch 의 원소는 jsonl 한 줄을 json.loads 한 dict 여야 합니다'
# 원본에서 다시 세어 대조한다(수를 손으로 적으면 여기서 걸린다)
assert by_rel == dict(Counter(r['relation'] for r in batch)), \
    'by_rel 은 batch 의 relation 을 빠짐없이 센 결과여야 합니다'
print('✅ 통과!')

## 1-2. 문서별 트리플 수와 허용 관계 비율
**배경**: 어느 문서에서 트리플이 많이 나왔는지, 그리고 **그 문서의 추출이 얼마나 규격에 맞는지**를 함께 보면 어떤 문서를 다시 뽑아야 할지 판단할 수 있습니다.

**요구사항**:
- `batch` 를 `doc_id` 별로 집계해 사전 **`by_doc`** 에 담으세요(키=문서 id, 값=트리플 수).
- 문서마다 **허용 관계(`RELATION_SIGNATURES`) 비율**을 사전 **`allowed_ratio`** 에 담으세요. 값은 `허용 관계 트리플 수 / 그 문서의 전체 트리플 수` 를 **소수 둘째 자리로 반올림**한 수입니다.

**예시**: `by_doc['PMC13495420']` 는 **5**, `allowed_ratio['PMC13495420']` 는 **0.8**, `allowed_ratio['PMC13493261']` 는 **1.0** 입니다.

> 이 배치는 세 값이 모두 딱 떨어져(`0.8`·`1.0`·`1.0`) 반올림을 해도 결과가 같습니다. 그래도 반올림을 거는 이유는, 문서가 늘어 `4/6` 같은 값이 나오면 `0.6666666666666666` 이 그대로 리포트에 실리기 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1-1 과 같은 방식으로 집계 키만 doc_id 로 바꾼다.
- 비율은 분자(허용 관계만 센 값)와 분모(전체)를 문서별로 따로 세어 나눈다.

세부구현:
1. doc_id 값을 세어 by_doc 을 만든다.
2. relation 이 RELATION_SIGNATURES 에 있는 행만 골라 doc_id 를 다시 센다(허용분 집계).
3. by_doc 의 문서마다 허용분 / 전체 를 계산하고 round(값, 2) 로 반올림해 allowed_ratio 에 담는다.
   3-1. 허용분 집계에 없는 문서는 0 으로 본다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert by_doc['PMC13495420'] == 5 and by_doc['PMC13493261'] == 4, \
    'doc_id 를 키로 세었는지 확인하세요'
assert by_doc['PMC13495535'] == 4, '규격 밖 관계도 그 문서의 트리플 수에는 들어갑니다'
assert allowed_ratio['PMC13495420'] == 0.8, \
    'PMC13495420 는 5개 중 4개만 허용 관계입니다. 반올림 자리수(둘째 자리)도 확인하세요'
assert allowed_ratio['PMC13493261'] == 1.0 and allowed_ratio['PMC13495535'] == 1.0, \
    '모든 문서가 allowed_ratio 에 있어야 합니다'
print('✅ 통과!')

---
# 2. 정제와 적재 준비

진단한 배치를 정제하고, 시그니처로 방향까지 검사한 뒤, 이름을 id 로 바꿔 `MERGE` 문으로 옮깁니다(교안_02 3-1).

## 2-1. 허용 관계만 남기고 중복 제거하기
**배경**: 원출력을 정제하려면 **규격 밖 관계를 버리고**(필터) **완전히 같은 트리플을 합칩니다**(중복 제거). 두 단계를 이어서 합니다.

**요구사항**:
- `batch` 에서 `relation` 이 **`RELATION_SIGNATURES`** 에 있는 트리플만 골라 리스트 **`allowed`** 에 담으세요.
- 그다음 `(subject, relation, object)` 기준으로 중복을 없앤 **집합** **`clean`** 을 만들고, 그 개수를 **`n_clean`** 에 담으세요.

**예시**: 허용 관계는 12개지만, 완전히 같은 트리플 하나가 합쳐져 **11개**가 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 relation 으로 거른 뒤, (subject, relation, object) 튜플의 set 으로 만든다.

세부구현:
1. relation 이 RELATION_SIGNATURES 에 있는 트리플만 리스트 컴프리헨션으로 골라 allowed 에 담는다.
2. allowed 의 각 트리플에서 (subject, relation, object) 튜플을 뽑아 집합 컴프리헨션으로 clean 을 만든다.
3. len(clean) 을 n_clean 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(allowed) == 12, '규격 밖 관계 하나가 빠졌는지 확인하세요(13건에서 12건이 됩니다)'
assert n_clean == 11, '허용 관계 12개에서 완전 중복 하나가 합쳐졌는지 확인하세요'
# 원본에서 다시 걸러 대조한다
assert allowed == [r for r in batch if r['relation'] in RELATION_SIGNATURES], \
    'allowed 는 batch 를 순서 그대로 거른 결과여야 합니다'
assert clean == {(r['subject'], r['relation'], r['object']) for r in allowed}, \
    'clean 은 allowed 의 세 칸 튜플 집합이어야 합니다'
print('✅ 통과!')

## 2-2. 시그니처로 방향까지 검사하기
**배경**: 2-1 의 정제는 관계 **이름**만 봤습니다. 그런데 시그니처는 이름뿐 아니라 **(주어 타입, 목적어 타입)** 까지 못 박은 계약입니다. 이름이 허용 목록에 있어도 방향이 거꾸로면 그것은 다른 사실입니다.

**요구사항**:
- 함수 **`signature_ok(row)`** 를 만드세요. `relation` 이 `RELATION_SIGNATURES` 에 있고 `subject_type`·`object_type` 이 그 시그니처의 두 타입과 **같으면** `True`, 아니면 `False` 입니다.
- `batch` 에서 이 검사를 통과하지 **못한** 트리플을 리스트 **`bad`** 에 담으세요.
- 아래 `flipped` 를 그대로 만들어 `signature_ok(flipped)` 를 출력하세요. 이 배치에는 방향이 뒤집힌 트리플이 한 건도 없어서, 검사가 도는지 보려면 일부러 뒤집은 한 줄이 필요합니다.

```python
flipped = {'subject': 'fatigue', 'subject_type': 'Symptom', 'relation': 'PRESENTS',
           'object': 'sarcoidosis', 'object_type': 'Disease'}
```

**예시**: `len(bad)` 는 **1** 이고(`CAUSES` 한 건), `signature_ok(flipped)` 는 **False** 입니다. `batch` 에 이미 있는 `sarcoidosis PRESENTS fatigue` 는 `True` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 관계 이름이 목록에 없으면 곧바로 거짓이다. 있으면 시그니처의 두 타입을 꺼내 행의 두 타입과 견준다.

세부구현:
1. relation 이 RELATION_SIGNATURES 에 없으면 False 를 돌려준다.
2. 있으면 값 세 칸 중 앞의 둘(주어 타입, 목적어 타입)을 꺼낸다.
3. 행의 subject_type·object_type 과 둘 다 같은지 한 번에 비교해 돌려준다.
4. batch 를 돌며 signature_ok 가 거짓인 행만 bad 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert signature_ok({'subject_type': 'Disease', 'relation': 'PRESENTS',
                     'object_type': 'Symptom'}) is True, \
    '시그니처를 지키는 트리플은 True 여야 합니다'
assert signature_ok({'subject_type': 'Symptom', 'relation': 'PRESENTS',
                     'object_type': 'Disease'}) is False, \
    '관계 이름만 보면 절반만 보는 것입니다. 두 타입까지 견주세요(방향이 뒤집힌 트리플)'
assert signature_ok({'subject_type': 'Compound', 'relation': 'CAUSES',
                     'object_type': 'Disease'}) is False, \
    '허용 목록에 없는 관계 이름은 그 자리에서 False 입니다'
assert bad == [r for r in batch if not signature_ok(r)] and len(bad) == 1, \
    'bad 에는 검사를 통과하지 못한 트리플만 담습니다(이 배치에서는 CAUSES 한 건)'
print('✅ 통과!')

## 2-3. 이름을 id 로 바꾸고 못 붙는 것 가려내기
**배경**: 정제한 트리플을 그래프에 넣으려면 이름을 **id** 로 바꿔야 합니다. 사전에 없는 이름은 버리지 않고 **후보**로 따로 모읍니다(교안의 `:Candidate`).

**요구사항**:
- `allowed` 의 주어·목적어에 나온 **서로 다른 (이름, 타입)** 쌍을 `lookup_id` 로 조회하세요(같은 이름이 여러 트리플에 나와도 한 번만 조회합니다). `lookup_id` 는 `(id, 사유)` 두 칸을 돌려주고, 못 찾으면 첫 칸이 `None` 입니다.
- id 를 찾은 것은 사전 **`resolved`** 에 `{이름: id}` 로 담으세요.
- 못 찾은 이름은 **정렬한 리스트** **`candidates`** 에 담으세요.

**예시**: `resolved['methotrexate']` 는 `'Compound::DB00563'` 이고, `candidates` 에는 `'Belimumab'` 이 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- LV1 3-1 과 같은 방식. 대상이 allowed 로 바뀌었을 뿐이다.

세부구현:
1. allowed 를 돌며 (subject, subject_type) 과 (object, object_type) 을 한 집합에 모은다.
2. 그 집합을 돌며 lookup_id 를 호출해 두 칸 중 첫 칸(id)만 받는다.
3. 결과가 있으면 resolved 사전에, 없으면 임시 리스트에 이름을 담는다.
4. 임시 리스트를 sorted 로 정렬해 candidates 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert resolved.get('methotrexate') == 'Compound::DB00563', \
    '제공된 lookup_id 를 그대로 쓰세요(이름을 임의로 손보면 사전에서 못 찾습니다)'
assert len(resolved) == 8, '사전에 붙는 이름은 8개입니다'
expected = ['BAFF', 'Belimumab', 'Rhodiola rosea', 'SLE', 'night sweats', 'sarcoidosis']
assert sorted(set(candidates)) == expected, \
    '못 찾은 이름만 담았는지 확인하세요(사전에 안 붙는 이름은 여섯입니다)'
assert candidates == expected, \
    'candidates 에 같은 이름이 여러 번 들어 있지 않은지 보세요. 한 이름이 여러 트리플에 나오면 ' \
    '조회도 여러 번 됩니다(서로 다른 쌍만 모으세요). 마지막에 sorted 로 정렬도 하세요'
print('✅ 통과!')

## 2-4. 트리플을 Cypher MERGE 문으로 (붙는 것과 후보를 갈라서)
**배경**: 이제 트리플을 `MERGE` 문으로 옮깁니다. 이름이 사전에 붙으면 **id 를 키로**, 붙지 않으면 **`:Candidate` 로 격리**합니다(키를 고르는 규칙은 교안_02 3-1 과 같습니다).

> **값을 넣는 방식은 교안과 다릅니다.** 교안_02 3-1 은 만든 문장을 그 자리에서 **실행**하므로 값을 `$이름` 파라미터로 넘겼습니다(`Parkinson's disease` 처럼 작은따옴표가 든 값에 문장이 깨지지 않게). 여기서는 실행하지 않고 **문장을 만들어 눈으로 보는 것**이 목적이라 값을 문자열에 그대로 박습니다. 실제로 그래프에 넣을 때는 교안 3-1 의 파라미터 방식을 쓰세요.

**요구사항**:
- 함수 **`node_pattern(alias, name, node_type)`** 를 만드세요. 노드 하나의 패턴 문자열을 돌려줍니다.
  - `resolved` 에 이름이 있으면 `(a:Compound {id: 'Compound::DB00563'})` 꼴
  - 없으면 `(a:Candidate {name: 'SLE', type: 'Disease'})` 꼴
- 그 함수로 `allowed` 의 각 트리플을 아래 형식의 `MERGE` 문 세 줄로 바꿔 리스트 **`merges`** 를 만드세요(주어는 별칭 `a`, 목적어는 `b`).

```text
MERGE (a:Compound {id: 'Compound::DB00563'})
MERGE (b:Disease {id: 'Disease::DOID:7148'})
MERGE (a)-[:TREATS]->(b)
```

**예시**: `len(merges)` 는 **12** 이고, 그중 **9개**에는 `:Candidate` 가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 노드 하나를 문자열로 만드는 함수를 먼저 만들고, 트리플 하나를 세 줄로 잇는 일은 그 함수를 두 번 부른다.

세부구현:
1. node_pattern 안에서 이름이 resolved 에 있는지 본다.
   1-1. 있으면 레이블은 node_type, 키는 id 하나다.
   1-2. 없으면 레이블은 Candidate, 키는 name 과 type 두 개다.
2. 트리플 하나를 MERGE 세 줄로 잇는 함수를 만든다(주어 패턴, 목적어 패턴, 관계).
3. 리스트 컴프리헨션으로 merges 를 만든다.
4. f-string 안에서 Cypher 의 중괄호는 두 번씩 쓴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(merges) == 12, '허용 관계 트리플 전부를 옮겼는지 확인하세요(중복은 그대로 둡니다)'
assert all(m.count('MERGE') == 3 for m in merges), '한 트리플당 MERGE 세 줄이 나와야 합니다'
assert sum(1 for m in merges if ':Candidate' in m) == 9, \
    '사전에 안 붙는 이름이 한쪽이라도 있으면 그 문장에는 :Candidate 가 들어갑니다'
assert node_pattern('a', 'methotrexate', 'Compound') == \
    "(a:Compound {id: 'Compound::DB00563'})", '붙는 이름은 id 를 키로 씁니다'
assert node_pattern('b', 'SLE', 'Disease') == \
    "(b:Candidate {name: 'SLE', type: 'Disease'})", \
    '안 붙는 이름은 :Candidate 에 name 과 type 두 키를 씁니다'
print('✅ 통과!')

## 2-5. 타입 계층으로 받아들일 개체 넓히기
**배경**: 이 배치의 `BAFF` 는 유전자라기보다 **사이토카인**이고, `SLE` 는 **자가면역질환**입니다. 더 좁게 부르고 싶은데 시그니처의 타입(`Gene`·`Disease`)은 건드리고 싶지 않을 때, 교안_02 3-3 의 **타입 계층**을 씁니다. 상위 타입 레이블을 함께 붙이면 시그니처는 그대로 두고 표현력만 넓어집니다.

**요구사항**:
- 사전 **`TYPE_HIERARCHY`** 를 만드세요. 키가 하위 타입, 값이 상위 타입입니다. `"Cytokine"` 의 상위로 `"Gene"` 을, `"Autoimmune"` 의 상위로 `"Disease"` 를 넣습니다.
- 함수 **`labels_for(node_type)`** 를 만드세요. 하위 타입이면 `[상위 타입, 그 타입]` 리스트를, 상위가 없으면 `[그 타입]` 리스트를 돌려줍니다(**상위 타입이 먼저**입니다).
- 함수 **`label_clause(node_type)`** 를 만드세요. `labels_for` 의 결과를 콜론(`:`)으로 이어 Cypher 에 그대로 넣을 문자열 하나로 돌려줍니다.

**예시**: `labels_for("Cytokine")` 는 `["Gene", "Cytokine"]`, `labels_for("Compound")` 는 `["Compound"]` 입니다. `label_clause("Cytokine")` 는 `"Gene:Cytokine"`, `label_clause("Compound")` 는 `"Compound"` 입니다(그래서 `(a:Gene:Cytokine)` 처럼 끼워 넣을 수 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 계층 사전에서 상위 타입을 찾아 보고, 있으면 두 개짜리, 없으면 한 개짜리 리스트를 돌려준다.
- 이어 붙이는 함수는 그 리스트를 구분자로 join 한다.

세부구현:
1. 하위 타입을 키로, 상위 타입을 값으로 담은 사전을 만든다.
2. labels_for 안에서 사전을 조회한다(없을 수 있으므로 기본값을 주는 조회를 쓴다).
3. 상위가 있으면 [상위, 자기], 없으면 [자기] 를 돌려준다. 순서는 상위가 먼저다.
4. label_clause 는 labels_for 결과를 콜론으로 join 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert TYPE_HIERARCHY["Cytokine"] == "Gene" and TYPE_HIERARCHY["Autoimmune"] == "Disease", \
    '계층 사전은 키=하위 타입, 값=상위 타입입니다'
assert labels_for("Cytokine") == ["Gene", "Cytokine"], '상위 타입이 먼저, 그다음이 자기 타입입니다'
assert labels_for("Compound") == ["Compound"], '상위가 없는 타입은 자기 자신 하나만 돌려줍니다'
assert label_clause("Cytokine") == "Gene:Cytokine", '레이블 여러 개는 콜론으로 잇습니다'
assert label_clause("Compound") == "Compound", '레이블이 하나면 콜론 없이 그 이름만 나옵니다'
# 사전을 하드코딩하지 않고 정말 조회하는지 본다(없던 하위 타입을 넣어 본다)
TYPE_HIERARCHY["Enzyme"] = "Gene"
assert labels_for("Enzyme") == ["Gene", "Enzyme"], \
    'labels_for 가 TYPE_HIERARCHY 를 조회하지 않고 값을 박아 둔 것으로 보입니다'
del TYPE_HIERARCHY["Enzyme"]
print('✅ 통과!')

---
# 3. 모델로 뽑기

여기서 처음 모델을 부릅니다. 제공된 프롬프트로 실제 추출을 돌리고, CoT 프롬프트를 직접 설계합니다(교안_02 2-1·2-2).

## 3-1. 실제 모델로 뽑고 그 자리에서 정제하기
**배경**: 준비 셀에서 제공한 **`extraction_chain`**(온톨로지 템플릿 + 구조화 출력 모델)으로 논문 발췌에서 트리플을 실제로 뽑습니다. 모델 답은 늘 규격에 맞지 않으므로, 뽑은 자리에서 바로 정제까지 합니다.

**요구사항**:
- 제공된 `extraction_chain` 을 `LIVE_TEXT` 로 호출한 결과를 **`live_result`** 에 담으세요(`invoke` 에 `{"text": LIVE_TEXT}` 를 넘깁니다).
- `live_result.triples` 를 순회하며 `(주어, 관계, 목적어)` 를 출력하세요.
- 근거가 그 트리플을 뒷받침하는지 보는 함수 **`grounded(tp, text)`** 를 만드세요. `tp.evidence` 가 `text` 안에 있고, `tp.subject` 와 `tp.object` 가 **둘 다** `tp.evidence` 안에 있으면 `True` 입니다(교안_02 3-1 의 후처리와 같은 판정).
- 허용 관계만 남기는 함수 **`keep_allowed(triples)`** 를 만드세요. `Triple` 객체 리스트를 받아 `relation` 이 **`RELATION_SIGNATURES`** 에 있는 것만 `(subject, relation, object)` 튜플의 **집합**으로 돌려줍니다.
- `live_result.triples` 중 `grounded(tp, LIVE_TEXT)` 가 `True` 인 것만 골라 `keep_allowed` 에 넣고, 그 결과를 **`live_clean`** 에 담으세요.

**예시**: `live_result` 는 `Extraction` 객체이고, 주어에 **Methotrexate**(또는 `MTX`)가 들어간 트리플이 있습니다. `live_clean` 은 원소가 1개 이상 8개 이하인 집합입니다. (구체적 개수·문구는 모델·실행에 따라 달라집니다.)

> 이 발췌의 답은 마침 관계가 전부 허용 목록 안에 있어, `live_clean` 만 봐서는 필터가 도는지 알 수 없습니다. 그래서 자가채점이 **규격 밖 관계를 섞은 가짜 트리플**을 `keep_allowed` 에 넣어 따로 확인합니다.

> 제공된 프롬프트를 **그대로** 호출하세요. 프롬프트를 바꾸면 아래 해설에 적힌 결과와 견줄 수 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공된 추출기에 제공된 프롬프트를 넘겨 결과 객체를 받는다. 파싱은 필요 없다.
- 정제는 2-1 과 같다. 다만 dict 가 아니라 Triple 객체라 점(.)으로 칸을 꺼낸다.

세부구현:
1. 추출기를 호출해 결과를 live_result 에 담는다.
2. live_result.triples 를 돌며 (주어, 관계, 목적어) 를 출력한다.
3. grounded 는 evidence 가 text 안에 있는지, 주어와 목적어가 evidence 안에 있는지 셋을 and 로 묶는다.
4. keep_allowed 안에서 relation 이 허용 관계인 것만 세 칸 튜플로 묶어 집합으로 돌려준다.
5. grounded 로 거른 트리플만 keep_allowed 에 넣어 결과를 live_clean 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert type(live_result).__name__ == 'Extraction', \
    '제공된 extraction_chain 을 그대로 호출했는지 확인하세요(구조화 출력이라 객체가 나옵니다)'
assert isinstance(live_result.triples, list) and 1 <= len(live_result.triples) <= 8, \
    '트리플이 하나도 없거나 지나치게 많습니다. LIVE_TEXT 를 그대로 넘겼는지 보세요'
assert any('ethotrexate' in tp.subject or 'MTX' in tp.subject for tp in live_result.triples), \
    '발췌의 주인공(메토트렉세이트)이 주어인 트리플이 하나도 없습니다'
assert isinstance(live_clean, set) and 1 <= len(live_clean) <= 8, \
    'live_clean 은 (subject, relation, object) 튜플의 집합이어야 합니다'
assert all(rel in RELATION_SIGNATURES for (_, rel, _) in live_clean), \
    'live_clean 에 허용 관계 밖 값이 남아 있습니다'

# 이 발췌의 답은 전부 허용 관계라, 필터를 안 걸어도 위 검사는 통과한다.
# 그래서 규격 밖 관계를 섞은 가짜 트리플을 넣어 필터가 정말 도는지 확인한다.
from types import SimpleNamespace

probe = [SimpleNamespace(subject='aspirin', relation='BINDS', object='PTGS1'),
         SimpleNamespace(subject='aspirin', relation='CAUSES', object='gastritis')]
assert keep_allowed(probe) == {('aspirin', 'BINDS', 'PTGS1')}, \
    '허용 관계가 아닌 CAUSES 가 걸러지지 않았습니다. 필터를 실제로 걸었는지 보세요'

# grounded 도 같은 방식으로 확인한다. 셋 다 다른 이유로 걸려야 한다
_txt = 'aspirin inhibits PTGS1 in platelets'
_ok = SimpleNamespace(subject='aspirin', object='PTGS1', evidence='aspirin inhibits PTGS1')
_fake = SimpleNamespace(subject='aspirin', object='PTGS1', evidence='aspirin blocks PTGS1')
_off = SimpleNamespace(subject='aspirin', object='COX2', evidence='aspirin inhibits PTGS1')
assert grounded(_ok, _txt) is True, '근거가 원문에 있고 두 끝이 그 안에 있으면 True 여야 합니다'
assert grounded(_fake, _txt) is False, '원문에 없는 근거(지어낸 문장)를 걸러야 합니다'
assert grounded(_off, _txt) is False, '근거는 진짜여도 목적어가 그 문장에 없으면 걸러야 합니다'
print('✅ 통과!')

## 3-2. CoT 프롬프트 설계하기
**배경**: 교안에서 `reasoning` 칸을 `triples` **앞**에 둔 CoT 서식과, 생각 순서를 단계로 적어 준 프롬프트 템플릿을 봤습니다. 같은 방식으로 **템플릿을 직접 설계**합니다.

**요구사항**:
- 함수 **`cot_prompt()`** 를 만드세요. 인자는 없고 **`ChatPromptTemplate`** 하나를 돌려줍니다.
- system 메시지에 아래를 빈 줄로 구분해 순서대로 담으세요(세부 문구는 자유).
  1. 역할 한 줄
  2. `build_ontology_block(RELATION_SIGNATURES)` 로 만든 온톨로지 블록
  3. 라벨 `[생각 순서]` 와 그 아래 **네 단계**. 각 줄은 `1)` `2)` `3)` `4)` 로 시작하며 순서는 개체 나열 → 관계 후보 → 시그니처 대조 → 최종 트리플입니다.
- human 메시지에는 라벨 `[문서]` 와 템플릿 변수 **`{text}`** 를 담으세요. 변수는 이 하나뿐입니다.

**예시**: `cot_prompt().input_variables` 는 `['text']` 이고, `cot_prompt().format_messages(text='Methotrexate ...')` 는 메시지 **두 개**를 돌려줍니다. 첫 메시지 내용에 `'[허용 관계]'`·`'[생각 순서]'` 와 `1)`~`4)` 로 시작하는 줄 **네 개**가 있고, 둘째 메시지 내용에 `'[문서]'` 와 문서 내용이 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공된 build_extraction_prompt 와 같은 구조에 [생각 순서] 덩어리 하나를 더 끼우는 모양이다.

세부구현:
1. 블록 생성 함수로 온톨로지 블록을 만든다.
2. 역할 + 블록 + [생각 순서] 네 단계를 빈 줄로 이어 system 문자열 하나를 만든다.
3. ChatPromptTemplate 의 from_messages 에 system 과 human 두 짝을 넘겨 return 한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
tpl = cot_prompt()
assert tpl.input_variables == ['text'], \
    f'템플릿 변수는 text 하나여야 합니다(지금 {tpl.input_variables})'
msgs = tpl.format_messages(text='Methotrexate suppresses immune response in patients with RA.')
assert len(msgs) == 2, f'system 과 human 두 메시지여야 합니다(지금 {len(msgs)}개)'
system_text, human_text = msgs[0].content, msgs[1].content
assert '[허용 관계]' in system_text and 'TREATS' in system_text, \
    'system 메시지에 온톨로지 블록이 빠졌습니다'
assert '[생각 순서]' in system_text, 'system 메시지에 [생각 순서] 라벨을 대괄호까지 그대로 넣으세요'
assert '[문서]' in human_text, 'human 메시지에 [문서] 라벨을 대괄호까지 그대로 넣으세요'
assert 'Methotrexate suppresses immune response in patients with RA.' in human_text, \
    'human 메시지의 {text} 자리에 문서가 채워져야 합니다'
steps = [line for line in system_text.splitlines() if line.strip()[:2] in ('1)', '2)', '3)', '4)')]
assert len(steps) == 4, f'번호 붙은 단계가 네 줄이어야 합니다(지금 {len(steps)}줄)'
print('✅ 통과!')

---
# 4. 설계 판단

오늘 내린 두 가지 판단을 자기 말로 적습니다(교안_01 2-2 · 교안_02 3-1).

## 4-1. 온톨로지 설계 판단 (서술형)
**배경**: 이 단원에서 두 가지 판단을 배웠습니다. **관계의 방향을 시그니처로 못 박는 것**과 **사전에 없는 이름을 후보로 격리하는 것**.

**요구사항**: 아래 markdown 셀에 두 가지를 **자신의 말로** 서술하세요. 정답 노트북의 모범 서술과 비교하세요.
1. `PRESENTS` 의 주어를 `Disease`, 목적어를 `Symptom` 으로 **고정**하면 어떤 이점이 있나요?
2. 사전에 없는 이름을 그래프에서 **빼 버리는 것**과 **후보로 남기는 것**은 무엇이 다른가요?

*(여기에 자신의 설명을 서술하세요. 1. 방향 고정의 이점, 2. 후보로 남기는 것의 뜻)*

---
수고했어요! LV2 에서 배치 집계·필터+정제·id 해소·Cypher 매핑·라이브 추출·CoT 프롬프트 설계를 **조합**했습니다. 여기서 만든 뼈대가 다음 단원들에서 **품질 측정**과 **엔티티 정규화**로 이어집니다.